## 載入模型以供後處理


In [ ]:
"""## 9.0. 載入模型以供後處理 - 修復版"""

import os
from bertopic import BERTopic
import pandas as pd
from tabulate import tabulate

class ModelLoader:
    """統一模型載入器"""

    def __init__(self, config):
        self.config = config
        self.models_dir = os.path.join(config['data']['base_path'], config['output']['models_dir'])
        self.results_dir = os.path.join(config['data']['base_path'], config['output']['results_dir'])
        self.loaded_models = {}
        self.enhanced_topics = {}
        self.model_stats = {}

    def check_model_files(self):
        """檢查模型檔案是否存在"""
        print("檢查模型檔案...")
        print("=" * 50)

        if not os.path.exists(self.models_dir):
            print(f"錯誤: 模型目錄不存在: {self.models_dir}")
            return False

        # 模型檔案對應
        model_files = {
            'operation_positive': 'operation_positive_model.pkl',
            'operation_negative': 'operation_negative_model.pkl',
            'ip_positive': 'ip_positive_model.pkl',
            'ip_negative': 'ip_negative_model.pkl',
            'balance_update': 'balance_update_model.pkl'
        }

        file_status = []
        all_exist = True

        for dataset_name, filename in model_files.items():
            model_path = os.path.join(self.models_dir, filename)
            exists = os.path.exists(model_path)

            if exists:
                file_size = os.path.getsize(model_path) / (1024*1024)  # MB
                status = f"存在 ({file_size:.1f}MB)"
            else:
                status = "不存在"
                all_exist = False

            file_status.append([
                self.config['data']['datasets'][dataset_name]['description'],
                filename,
                status
            ])

        headers = ['資料集', '檔案名稱', '狀態']
        print(tabulate(file_status, headers=headers, tablefmt="grid"))

        if not all_exist:
            print(f"\n錯誤: 部分模型檔案缺失，請先執行模型訓練步驟 (8.1-8.5)")
            return False

        print(f"\n所有模型檔案都存在")
        return True

    def load_single_model(self, dataset_name, filename):
        """載入單個模型"""
        model_path = os.path.join(self.models_dir, filename)

        try:
            # 載入模型
            model = BERTopic.load(model_path)

            # 重新設定嵌入模型 (重要!)
            if not hasattr(model, 'embedding_model') or model.embedding_model is None:
                if 'analyzer' in globals() and hasattr(globals()['analyzer'], 'embedding_model'):
                    model.embedding_model = globals()['analyzer'].embedding_model
                    print(f"  已重新設定嵌入模型")
                else:
                    print(f"  警告: 無法設定嵌入模型，analyzer不存在")

            # 獲取模型統計資訊
            topic_info = model.get_topic_info()
            total_docs = topic_info['Count'].sum()
            valid_topics = len(topic_info[topic_info['Topic'] != -1])
            outlier_count = topic_info[topic_info['Topic'] == -1]['Count'].values[0] if -1 in topic_info['Topic'].values else 0
            coverage_rate = (total_docs - outlier_count) / total_docs * 100

            # 儲存統計資訊
            self.model_stats[dataset_name] = {
                'total_docs': total_docs,
                'valid_topics': valid_topics,
                'outlier_count': outlier_count,
                'coverage_rate': coverage_rate,
                'largest_topic_size': topic_info[topic_info['Topic'] != -1]['Count'].max() if valid_topics > 0 else 0
            }

            # 儲存到全域變數和本地 - 關鍵修復點
            model_var = f'{dataset_name}_model'
            globals()[model_var] = model
            self.loaded_models[dataset_name] = model

            return True, f"成功 ({total_docs}文檔, {valid_topics}主題)"

        except Exception as e:
            return False, f"失敗: {str(e)}"

    def load_all_models(self):
        """載入所有模型"""
        print(f"\n載入模型...")
        print("=" * 50)

        model_files = {
            'operation_positive': 'operation_positive_model.pkl',
            'operation_negative': 'operation_negative_model.pkl',
            'ip_positive': 'ip_positive_model.pkl',
            'ip_negative': 'ip_negative_model.pkl',
            'balance_update': 'balance_update_model.pkl'
        }

        loading_results = []
        success_count = 0

        for dataset_name, filename in model_files.items():
            print(f"載入 {dataset_name}...")
            success, message = self.load_single_model(dataset_name, filename)

            if success:
                success_count += 1

            loading_results.append([
                self.config['data']['datasets'][dataset_name]['description'],
                message
            ])

        print(f"\n載入結果:")
        headers = ['資料集', '載入狀態']
        print(tabulate(loading_results, headers=headers, tablefmt="grid"))

        print(f"\n成功載入 {success_count}/{len(model_files)} 個模型")
        return success_count > 0

    def check_enhanced_topics(self):
        """檢查已有的LLM增強主題"""
        print(f"\n檢查主題命名狀態...")
        print("=" * 50)

        enhanced_status = []

        for dataset_name in self.loaded_models.keys():
            # 檢查各種可能的增強主題變數
            naming_status = []

            # 檢查人工命名 (優先)
            manual_var = f'{dataset_name}_enhanced'
            if manual_var in globals() and globals()[manual_var]:
                manual_count = len(globals()[manual_var])
                naming_status.append(f"人工: {manual_count}個")
                # 儲存到本地
                self.enhanced_topics[dataset_name] = globals()[manual_var]

            # 檢查LLM命名
            llm_var = f'{dataset_name}_enhanced_llm'
            if llm_var in globals() and globals()[llm_var]:
                llm_count = len(globals()[llm_var])
                naming_status.append(f"LLM: {llm_count}個")
                # 如果沒有人工命名，則使用LLM命名
                if dataset_name not in self.enhanced_topics:
                    self.enhanced_topics[dataset_name] = globals()[llm_var]

            # 檢查手動增強主題檔案
            manual_enhanced_path = os.path.join(self.results_dir, f'{dataset_name}_manual_enhanced_topics.json')
            manual_file_exists = os.path.exists(manual_enhanced_path)

            status_str = "; ".join(naming_status) if naming_status else "未命名"
            file_status = "存在" if manual_file_exists else "不存在"

            enhanced_status.append([
                self.config['data']['datasets'][dataset_name]['description'],
                status_str,
                file_status
            ])

        headers = ['資料集', '記憶體命名', '人工命名檔案']
        print(tabulate(enhanced_status, headers=headers, tablefmt="grid"))

    def load_manual_enhanced_topics_from_files(self):
        """從檔案載入人工增強主題"""
        print(f"\n載入已保存的人工命名...")
        print("=" * 50)

        loaded_count = 0

        for dataset_name in self.loaded_models.keys():
            json_path = os.path.join(self.results_dir, f'{dataset_name}_manual_enhanced_topics.json')

            if os.path.exists(json_path):
                try:
                    import json
                    with open(json_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)

                    # 轉換回整數key
                    enhanced_topics = {int(k): v for k, v in data.items()}

                    # 載入到全域變數 - 關鍵修復點
                    globals()[f'{dataset_name}_enhanced'] = enhanced_topics
                    self.enhanced_topics[dataset_name] = enhanced_topics

                    print(f"已載入 {dataset_name}: {len(enhanced_topics)} 個人工命名")
                    loaded_count += 1

                except Exception as e:
                    print(f"載入 {dataset_name} 人工命名失敗: {str(e)}")
            else:
                print(f"{dataset_name}: 無人工命名檔案")

        print(f"\n成功載入 {loaded_count} 個人工命名檔案")

    def generate_model_summary(self):
        """生成模型統計摘要"""
        print(f"\n模型統計摘要")
        print("=" * 50)

        if not self.model_stats:
            print("錯誤: 沒有載入的模型統計資訊")
            return

        summary_data = []
        total_docs = 0
        total_topics = 0

        for dataset_name, stats in self.model_stats.items():
            # 檢查是否有增強主題
            enhanced_count = 0
            enhanced_source = "無"
            if dataset_name in self.enhanced_topics:
                enhanced_count = len(self.enhanced_topics[dataset_name])
                # 檢查來源
                if f'{dataset_name}_enhanced' in globals() and globals()[f'{dataset_name}_enhanced']:
                    enhanced_source = "人工"
                elif f'{dataset_name}_enhanced_llm' in globals() and globals()[f'{dataset_name}_enhanced_llm']:
                    enhanced_source = "LLM"

            summary_data.append([
                self.config['data']['datasets'][dataset_name]['description'],
                stats['total_docs'],
                stats['valid_topics'],
                f"{stats['outlier_count']} ({stats['outlier_count']/stats['total_docs']*100:.1f}%)",
                f"{stats['coverage_rate']:.1f}%",
                stats['largest_topic_size'],
                f"{enhanced_count}個({enhanced_source})"
            ])

            total_docs += stats['total_docs']
            total_topics += stats['valid_topics']

        headers = ['資料集', '總文檔', '主題數', '離群文檔', '覆蓋率', '最大主題', '已命名']
        print(tabulate(summary_data, headers=headers, tablefmt="grid"))

        print(f"\n總計: {total_docs:,} 個文檔，{total_topics} 個主題")

    def verify_all_components(self):
        """驗證所有組件是否正常"""
        print(f"\n驗證系統組件...")
        print("=" * 50)

        verification_results = []

        # 檢查分析器
        analyzer_status = "正常" if 'analyzer' in globals() and hasattr(globals()['analyzer'], 'embedding_model') and globals()['analyzer'].embedding_model is not None else "異常"
        verification_results.append(['文本分析器', analyzer_status])

        # 檢查訓練器
        trainer_status = "正常" if 'trainer' in globals() else "異常"
        verification_results.append(['模型訓練器', trainer_status])

        # 檢查增強器
        enhancer_status = "正常" if 'enhancer' in globals() and hasattr(globals()['enhancer'], 'client') and globals()['enhancer'].client is not None else "異常"
        verification_results.append(['LLM增強器', enhancer_status])

        # 檢查評估器
        evaluator_status = "正常" if 'evaluator' in globals() else "異常"
        verification_results.append(['模型評估器', evaluator_status])

        # 檢查視覺化器
        visualizer_status = "正常" if 'visualizer' in globals() else "異常"
        verification_results.append(['視覺化器', visualizer_status])

        # 檢查人工命名器
        manual_namer_status = "正常" if 'manual_namer' in globals() else "異常"
        verification_results.append(['人工命名器', manual_namer_status])

        headers = ['組件', '狀態']
        print(tabulate(verification_results, headers=headers, tablefmt="grid"))

    def verify_loaded_models(self):
        """驗證載入的模型"""
        print(f"\n驗證載入的模型...")
        print("=" * 50)

        model_verification = []

        expected_models = ['operation_positive_model', 'operation_negative_model',
                          'ip_positive_model', 'ip_negative_model', 'balance_update_model']

        for model_var in expected_models:
            if model_var in globals():
                model = globals()[model_var]

                # 檢查模型是否有必要的屬性
                has_embedding = hasattr(model, 'embedding_model') and model.embedding_model is not None
                has_topics = hasattr(model, 'topics_') and model.topics_ is not None
                has_get_topic = hasattr(model, 'get_topic')

                status = "完整" if all([has_embedding, has_topics, has_get_topic]) else "不完整"
                details = []
                if not has_embedding: details.append("缺少嵌入模型")
                if not has_topics: details.append("缺少主題分配")
                if not has_get_topic: details.append("缺少get_topic方法")

                detail_str = "; ".join(details) if details else "正常"

                model_verification.append([
                    model_var.replace('_model', ''),
                    status,
                    detail_str
                ])
            else:
                model_verification.append([
                    model_var.replace('_model', ''),
                    "未載入",
                    "變數不存在"
                ])

        headers = ['模型', '狀態', '詳情']
        print(tabulate(model_verification, headers=headers, tablefmt="grid"))

    def run_complete_loading(self):
        """執行完整載入流程"""
        print("開始模型載入與系統驗證")
        print("=" * 80)

        # 1. 檢查模型檔案
        if not self.check_model_files():
            return False

        # 2. 載入所有模型
        if not self.load_all_models():
            return False

        # 3. 載入人工增強主題檔案
        self.load_manual_enhanced_topics_from_files()

        # 4. 檢查增強主題
        self.check_enhanced_topics()

        # 5. 生成統計摘要
        self.generate_model_summary()

        # 6. 驗證載入的模型
        self.verify_loaded_models()

        # 7. 驗證系統組件
        self.verify_all_components()

        print(f"\n載入完成 系統已準備就緒")
        print("=" * 80)
        print("後續可用功能:")
        print("- 9.1 模型主題關鍵詞分析")
        print("- 9.2 LLM主題命名")
        print("- 9.3 人工主題命名")
        print("- 10. 模型評估分析")
        print("- 11. 視覺化分析")
        print("\n快速功能:")
        print("- quick_review_all() - 檢視所有主題")
        print("- quick_rename('dataset_name') - 人工重命名")
        print("- generate_visualizations() - 生成視覺化圖表")

        return True

def debug_global_variables():
    """調試全域變數狀態"""
    print("調試: 檢查全域變數狀態")
    print("="*50)

    # 檢查模型變數
    expected_models = ['operation_positive_model', 'operation_negative_model',
                      'ip_positive_model', 'ip_negative_model', 'balance_update_model']

    print("模型變數:")
    for model_var in expected_models:
        status = "存在" if model_var in globals() else "不存在"
        print(f"  {model_var}: {status}")

    # 檢查增強主題變數
    datasets = ['operation_positive', 'operation_negative', 'ip_positive', 'ip_negative', 'balance_update']
    print("\n增強主題變數:")
    for dataset in datasets:
        enhanced_var = f'{dataset}_enhanced'
        llm_var = f'{dataset}_enhanced_llm'

        enhanced_status = "存在" if enhanced_var in globals() and globals()[enhanced_var] else "不存在"
        llm_status = "存在" if llm_var in globals() and globals()[llm_var] else "不存在"

        enhanced_count = len(globals()[enhanced_var]) if enhanced_var in globals() and globals()[enhanced_var] else 0
        llm_count = len(globals()[llm_var]) if llm_var in globals() and globals()[llm_var] else 0

        print(f"  {enhanced_var}: {enhanced_status} ({enhanced_count}個)")
        print(f"  {llm_var}: {llm_status} ({llm_count}個)")

    # 檢查其他重要變數
    important_vars = ['analyzer', 'trainer', 'enhancer', 'evaluator', 'visualizer', 'manual_namer', 'PIPELINE_CONFIG']
    print("\n其他重要變數:")
    for var in important_vars:
        status = "存在" if var in globals() else "不存在"
        print(f"  {var}: {status}")

def load_saved_models():
    """載入雲端硬碟中已儲存的模型 - 簡化版"""
    print("載入已儲存的模型...")
    print("=" * 40)

    if 'PIPELINE_CONFIG' not in globals():
        print("錯誤: PIPELINE_CONFIG 未定義")
        return {}

    models_dir = os.path.join(PIPELINE_CONFIG['data']['base_path'], PIPELINE_CONFIG['output']['models_dir'])

    # 模型檔案對應
    model_files = {
        'operation_positive': 'operation_positive_model.pkl',
        'operation_negative': 'operation_negative_model.pkl',
        'ip_positive': 'ip_positive_model.pkl',
        'ip_negative': 'ip_negative_model.pkl',
        'balance_update': 'balance_update_model.pkl'
    }

    loaded_models = {}

    for dataset_name, filename in model_files.items():
        model_path = os.path.join(models_dir, filename)
        model_var = f'{dataset_name}_model'

        # 檢查檔案是否存在
        if os.path.exists(model_path):
            try:
                # 載入模型
                model = BERTopic.load(model_path)

                # 確保有嵌入模型
                if not hasattr(model, 'embedding_model') or model.embedding_model is None:
                    if 'analyzer' in globals() and hasattr(globals()['analyzer'], 'embedding_model'):
                        model.embedding_model = globals()['analyzer'].embedding_model
                        print(f"  {dataset_name}: 已重新設定嵌入模型")

                # 儲存到全域變數 - 關鍵修復點
                globals()[model_var] = model
                loaded_models[dataset_name] = model

                print(f"已載入 {dataset_name}: {filename}")

            except Exception as e:
                print(f"載入 {dataset_name} 失敗: {str(e)}")
        else:
            print(f"找不到 {dataset_name}: {filename}")

    print(f"\n成功載入 {len(loaded_models)} 個模型")
    return loaded_models

def enhance_all_topics():
    """為所有已訓練的模型進行主題命名 - 修復版"""

    # 檢查 API Key
    if 'PIPELINE_CONFIG' not in globals():
        print("錯誤: PIPELINE_CONFIG 未定義")
        return None

    if PIPELINE_CONFIG['llm']['api_key'] is None:
        print("請在 Colab Secrets 中設定 OPENAI_API_KEY")
        print("或手動設定: PIPELINE_CONFIG['llm']['api_key'] = 'your_api_key_here'")
        return None

    print("開始進行主題命名...")
    print("=" * 40)

    # 首先嘗試載入已儲存的模型
    print("檢查並載入已儲存的模型...")
    load_saved_models()

    enhanced_results = {}

    # 檢查並處理每個模型
    model_vars = {
        'operation_positive': 'operation_positive_model',
        'operation_negative': 'operation_negative_model',
        'ip_positive': 'ip_positive_model',
        'ip_negative': 'ip_negative_model',
        'balance_update': 'balance_update_model'
    }

    for dataset_name, model_var in model_vars.items():
        if model_var in globals():
            print(f"\n命名 {dataset_name} 主題...")
            model = globals()[model_var]

            # 檢查enhancer是否存在
            if 'enhancer' not in globals():
                print("錯誤: enhancer 未初始化")
                continue

            enhanced = globals()['enhancer'].enhance_model_topics(model, dataset_name)
            enhanced_results[dataset_name] = enhanced

            # 儲存增強結果到全域變數 - 使用一致的命名，關鍵修復點
            globals()[f'{dataset_name}_enhanced'] = enhanced
            globals()[f'{dataset_name}_enhanced_llm'] = enhanced  # 備份
            print(f"命名結果已儲存至 {dataset_name}_enhanced 和 {dataset_name}_enhanced_llm")
        else:
            print(f"模型 {model_var} 未找到，跳過")

    print(f"\n主題命名完成，共處理 {len(enhanced_results)} 個模型")

    # 驗證結果
    print("\n驗證命名結果:")
    for dataset_name, enhanced in enhanced_results.items():
        var_name = f'{dataset_name}_enhanced'
        if var_name in globals() and globals()[var_name]:
            print(f"  {dataset_name}: 成功保存 {len(globals()[var_name])} 個主題命名")
        else:
            print(f"  {dataset_name}: 保存失敗")

    return enhanced_results

# 初始化載入器
if 'PIPELINE_CONFIG' in globals():
    model_loader = ModelLoader(PIPELINE_CONFIG)
    print("ModelLoader 已初始化")
else:
    print("錯誤: PIPELINE_CONFIG 未定義，無法初始化 ModelLoader")

def run_model_loading():
    """執行完整載入流程的便捷函數"""
    if 'model_loader' not in globals():
        print("錯誤: ModelLoader 未初始化")
        return False

    print("統一模型載入與驗證")
    print("="*80)

    # 先調試檢查變數狀態
    debug_global_variables()

    success = model_loader.run_complete_loading()

    if success:
        print(f"\n模型載入成功 現在可以進行後續分析了")
    else:
        print(f"\n模型載入失敗，請檢查模型訓練步驟是否已完成")

    return success

# 執行完整載入流程
print("統一模型載入與驗證")
print("="*80)

# 先進行調試
debug_global_variables()

# 執行載入
if 'PIPELINE_CONFIG' in globals() and 'model_loader' in globals():
    success = model_loader.run_complete_loading()

    if success:
        print(f"\n模型載入成功 現在可以進行後續分析了")
    else:
        print(f"\n模型載入失敗，請檢查模型訓練步驟是否已完成")
else:
    print("錯誤: 缺少必要的全域變數，請確保已執行前面的初始化步驟")

ModelLoader 已初始化
統一模型載入與驗證
調試: 檢查全域變數狀態
模型變數:
  operation_positive_model: 不存在
  operation_negative_model: 不存在
  ip_positive_model: 不存在
  ip_negative_model: 不存在
  balance_update_model: 不存在

增強主題變數:
  operation_positive_enhanced: 不存在 (0個)
  operation_positive_enhanced_llm: 不存在 (0個)
  operation_negative_enhanced: 不存在 (0個)
  operation_negative_enhanced_llm: 不存在 (0個)
  ip_positive_enhanced: 不存在 (0個)
  ip_positive_enhanced_llm: 不存在 (0個)
  ip_negative_enhanced: 不存在 (0個)
  ip_negative_enhanced_llm: 不存在 (0個)
  balance_update_enhanced: 不存在 (0個)
  balance_update_enhanced_llm: 不存在 (0個)

其他重要變數:
  analyzer: 存在
  trainer: 存在
  enhancer: 存在
  evaluator: 不存在
  visualizer: 存在
  manual_namer: 不存在
  PIPELINE_CONFIG: 存在
開始模型載入與系統驗證
檢查模型檔案...
+------------------+------------------------------+----------------+
| 資料集           | 檔案名稱                     | 狀態           |
+==================+==============================+================+
| 營運活動正面情緒 | operation_positive_model.pkl | 存在 (847.8MB) |
+----------